# LangChain RAG System - Interactive Demo

This notebook demonstrates the key features of our RAG system:
- Document loading and chunking
- Vector embeddings with Ollama
- Semantic search with ChromaDB
- RAG chain with LCEL
- Token-level streaming
- LangSmith tracing

## Setup

In [ ]:
# Import necessary modules
import sys
sys.path.append('..')  # Add parent directory to path

from src.config import config
from src.document_loader import DocumentLoader
from src.vector_store import VectorStore
from src.rag_chain import RAGChain
from src.streaming import format_streaming_response

print("✅ Imports successful")
print(f"Using model: {config.ollama_model}")
print(f"ChromaDB directory: {config.chroma_persist_directory}")

## Step 1: Load and Process Documents

In [ ]:
# Initialize document loader
doc_loader = DocumentLoader()

# Load PDFs from data/papers directory
# Make sure you have PDF files in ../data/papers/
chunks = doc_loader.load_and_split("../data/papers/")

print(f"\n📊 Loaded {len(chunks)} chunks")

# Show statistics
stats = doc_loader.get_chunk_stats(chunks)
print(f"\nChunk Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Examine a sample chunk
if chunks:
    sample = chunks[0]
    print("Sample Chunk:")
    print("=" * 60)
    print(f"Source: {sample.metadata.get('source', 'Unknown')}")
    print(f"Page: {sample.metadata.get('page', 'N/A')}")
    print(f"\nContent preview (first 500 chars):")
    print(sample.page_content[:500] + "...")

## Step 2: Create Vector Store

In [ ]:
# Initialize vector store
vector_store = VectorStore()

# Create embeddings and store in ChromaDB
# This step may take a few minutes depending on document size
print("Creating embeddings... This may take a few minutes.")
vectorstore = vector_store.create_vectorstore(chunks)

print("\n✅ Vector store created successfully!")

## Step 3: Test Similarity Search

In [ ]:
# Test similarity search
test_query = "What are the main findings?"

print(f"Query: {test_query}")
print("\n" + "=" * 60)

# Search for similar documents
results = vector_store.similarity_search(test_query, k=3)

print(f"\nFound {len(results)} similar documents:\n")

for i, doc in enumerate(results, 1):
    source = doc.metadata.get('source', 'Unknown').split('/')[-1]
    page = doc.metadata.get('page', 'N/A')
    
    print(f"{i}. {source} (page {page})")
    print(f"   Preview: {doc.page_content[:150]}...")
    print()

## Step 4: Create RAG Chain

In [ ]:
# Create RAG chain
rag_chain = RAGChain(vector_store)

print("✅ RAG chain created with streaming support")

## Step 5: Query with Non-Streaming Mode

In [ ]:
# Test non-streaming query
question = "What is the main contribution of this research?"

print(f"Question: {question}")
print("\n" + "=" * 60)
print("Answer (non-streaming):")
print("=" * 60)

response = rag_chain.invoke(question)
print(response)

# Get sources
sources = rag_chain.get_sources(question)
print("\n" + "=" * 60)
print(f"Sources ({len(sources)} documents):")
print("=" * 60)
for i, doc in enumerate(sources, 1):
    source = doc.metadata.get('source', 'Unknown').split('/')[-1]
    page = doc.metadata.get('page', 'N/A')
    print(f"{i}. {source} (page {page})")

## Step 6: Query with Streaming Mode

In [ ]:
# Test streaming query
question = "What methods were used in this research?"

print(f"Question: {question}")
print("\n" + "=" * 60)
print("Answer (streaming):")
print("=" * 60)

# Stream response
full_response = ""
for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)
    full_response += chunk

print("\n")

# Get sources
sources = rag_chain.get_sources(question)
print("=" * 60)
print(f"Sources ({len(sources)} documents):")
print("=" * 60)
for i, doc in enumerate(sources, 1):
    source = doc.metadata.get('source', 'Unknown').split('/')[-1]
    page = doc.metadata.get('page', 'N/A')
    print(f"{i}. {source} (page {page})")

## Step 7: Interactive Q&A

In [ ]:
# Interactive query function
def ask_question(question: str):
    """Helper function for interactive queries."""
    print(f"\n💬 Question: {question}")
    print("\n🤖 Answer: ", end="", flush=True)
    
    # Stream response
    for chunk in rag_chain.stream(question):
        print(chunk, end="", flush=True)
    
    print("\n")
    
    # Show sources
    sources = rag_chain.get_sources(question)
    print(f"📚 Sources: {len(sources)} documents")
    for i, doc in enumerate(sources, 1):
        source = doc.metadata.get('source', 'Unknown').split('/')[-1]
        page = doc.metadata.get('page', 'N/A')
        print(f"   {i}. {source} (page {page})")

# Try different questions
ask_question("What are the key findings?")

In [ ]:
# Ask another question
ask_question("What are the limitations of this approach?")

## Step 8: Experiment with Different Parameters

In [ ]:
# Try different retrieval K values
print("Testing with different K values...\n")

question = "What is discussed in the paper?"

for k in [2, 4, 6]:
    print(f"\n{'=' * 60}")
    print(f"Retrieval K = {k}")
    print("=" * 60)
    
    # Update retrieval parameter
    rag_chain.update_retrieval_k(k)
    
    # Get sources to see how many are retrieved
    sources = rag_chain.get_sources(question)
    print(f"Retrieved {len(sources)} documents")
    
    # Generate answer
    response = rag_chain.invoke(question)
    print(f"\nAnswer length: {len(response)} characters")
    print(f"Preview: {response[:200]}...")

## Step 9: Load Existing Vector Store

In [ ]:
# Demonstrate loading existing vector store
# This is useful when you restart the notebook

print("Loading existing vector store...")

new_vector_store = VectorStore()
if new_vector_store.load_vectorstore():
    print("✅ Successfully loaded existing vector store")
    
    # Create new RAG chain
    new_rag_chain = RAGChain(new_vector_store)
    
    # Test query
    test_q = "Summarize the main points"
    print(f"\nTest query: {test_q}")
    print("\nAnswer: ", end="", flush=True)
    
    for chunk in new_rag_chain.stream(test_q):
        print(chunk, end="", flush=True)
    print()
else:
    print("❌ No existing vector store found")

## Summary

This notebook demonstrated:

1. **Document Loading**: Loading and chunking PDF documents
2. **Vector Store**: Creating embeddings and storing in ChromaDB
3. **Similarity Search**: Finding relevant document chunks
4. **RAG Chain**: Building a retrieval-augmented generation pipeline
5. **Streaming**: Token-level response streaming for better UX
6. **Parameter Tuning**: Experimenting with different retrieval settings
7. **Persistence**: Loading existing vector stores

### Next Steps

- Experiment with different chunk sizes and overlaps
- Try different embedding models
- Test various prompt templates
- Evaluate retrieval quality
- Add conversation memory

### Resources

- [LangChain Documentation](https://python.langchain.com)
- [LangSmith Tracing](https://smith.langchain.com)
- [ChromaDB Docs](https://docs.trychroma.com)
- [Ollama Models](https://ollama.ai/library)